# ENTREGA 2 — Perfilado, Diccionario y Limpieza Inicial
## DATA VISUALIZATION (1ACC0211) — Universidad Peruana de Ciencias Aplicadas

**Proyecto:** Dinámica del comercio mundial: patrones de exportación e importación por país, categoría de producto y región geográfica (1989–2023)

| Código | Nombre |
|---|---|
| U202218912 | Julio Cesar Meza Alfaro |
| U202212675 | Rosa María Rodríguez Valencia |
| U202214069 | Braulio Alonso Bartra Sandoval |

---

### Estructura del notebook
1. Configuración e importaciones
2. Carga y exploración inicial
3. Unidad de análisis
4. Diccionario de datos
5. Tabla de perfilado completo
6. Identificación de problemas de calidad
7. Reglas de limpieza y transformaciones
8. Bitácora de transformaciones
9. Dataset limpio — validación y exportación
10. Modelado de datos

---
## 1. Configuración e importaciones

In [154]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 80)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 200)

RUTA_ORIGINAL = '../../data/raw/34_years_world_export_import_dataset.csv'
RUTA_LIMPIO   = '../../data/processed/dataset_limpio_entrega2.csv'

print('Librerías cargadas correctamente.')
print(f'Pandas {pd.__version__} | NumPy {np.__version__}')

Librerías cargadas correctamente.
Pandas 3.0.2 | NumPy 2.4.4


---
## 2. Carga y exploración inicial

In [155]:
df = pd.read_csv(RUTA_ORIGINAL)

print('=' * 60)
print('DIMENSIONES DEL DATASET')
print('=' * 60)
print(f'  Filas    : {df.shape[0]:,}')
print(f'  Columnas : {df.shape[1]}')
print(f'  Memoria  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print()
print('RANGO TEMPORAL')
print(f'  Año mínimo : {df["Year"].min()}')
print(f'  Año máximo : {df["Year"].max()}')
print(f'  Años únicos: {df["Year"].nunique()}')
print()
print('COBERTURA GEOGRÁFICA')
print(f'  Países / territorios únicos: {df["Partner Name"].nunique()}')

DIMENSIONES DEL DATASET
  Filas    : 8,096
  Columnas : 33
  Memoria  : 2.43 MB

RANGO TEMPORAL
  Año mínimo : 1988
  Año máximo : 2021
  Años únicos: 34

COBERTURA GEOGRÁFICA
  Países / territorios únicos: 265


In [156]:
print('PRIMERAS 5 FILAS')
df.head()

PRIMERAS 5 FILAS


,Partner Name,Year,Export (US$ Thousand),Import (US$ Thousand),Export Product Share (%),Import Product Share (%),Revealed comparative advantage,World Growth (%),Country Growth (%),AHS Simple Average (%),AHS Weighted Average (%),AHS Total Tariff Lines,AHS Dutiable Tariff Lines Share (%),AHS Duty Free Tariff Lines Share (%),AHS Specific Tariff Lines Share (%),AHS AVE Tariff Lines Share (%),AHS MaxRate (%),AHS MinRate (%),AHS SpecificDuty Imports (US$ Thousand),AHS Dutiable Imports (US$ Thousand),AHS Duty Free Imports (US$ Thousand),MFN Simple Average (%),MFN Weighted Average (%),MFN Total Tariff Lines,MFN Dutiable Tariff Lines Share (%),MFN Duty Free Tariff Lines Share (%),MFN Specific Tariff Lines Share (%),MFN AVE Tariff Lines Share (%),MFN MaxRate (%),MFN MinRate (%),MFN SpecificDuty Imports (US$ Thousand),MFN Dutiable Imports (US$ Thousand),MFN Duty Free Imports (US$ Thousand)
0,Aruba,1988,"3,498.10",328.49,100.00,100,NaN,NaN,NaN,2.80,2.92,155.00,18.06,60.00,20.00,1.94,50.00,0.00,"1,867.00","2,346.37",781.65,13.59,8.46,"1,152.00",63.54,22.74,70.32,31.61,352.69,0.00,"2,186.00","3,128.02",0.00
1,Afghanistan,1988,"213,030.40","54,459.52",100.00,100,NaN,NaN,NaN,0.88,1.83,548.00,8.76,82.66,8.03,0.55,35.00,0.00,"30,863.03","70,204.13","23,987.37",17.68,12.43,"4,142.00",69.41,15.64,72.45,40.51,"2,029.66",0.00,"78,436.91","94,191.50",0.00
2,Angola,1988,"375,527.89","370,702.76",100.00,100,NaN,NaN,NaN,2.02,3.89,633.00,25.43,69.19,5.37,0.00,40.00,0.00,"723,819.51","754,183.84","167,297.68",12.70,6.14,"5,438.00",76.00,16.27,41.55,24.80,451.15,0.00,"727,741.99","921,481.52",0.00
3,Anguila,1988,366.98,4.00,100.00,100,NaN,NaN,NaN,3.71,1.09,33.00,6.06,72.73,21.21,0.00,35.00,0.00,60.00,65.00,518.00,16.63,14.75,322.00,66.15,22.05,78.79,36.36,100.00,0.00,94.00,583.00,0.00
4,Albania,1988,"30,103.56","47,709.30",100.00,100,NaN,NaN,NaN,1.84,2.38,744.00,20.83,60.48,17.61,1.08,25.00,0.00,"18,806.15","62,294.53","38,901.42",19.20,9.68,"5,684.00",66.87,19.19,57.93,48.52,"3,000.00",0.00,"37,904.09","101,195.95",0.00


In [157]:
print('TIPOS DE DATOS')
df.dtypes.to_frame('Tipo')

TIPOS DE DATOS


,Tipo
Partner Name,str
Year,int64
Export (US$ Thousand),float64
Import (US$ Thousand),float64
Export Product Share (%),float64
Import Product Share (%),int64
Revealed comparative advantage,float64
World Growth (%),float64
Country Growth (%),float64
AHS Simple Average (%),float64


---
## 3. Unidad de análisis

La **unidad de análisis** del dataset es:

> **Un país (o territorio) en un año calendario determinado.**

Cada fila representa el flujo comercial agregado de un país o territorio específico para un año, incluyendo el valor total de exportaciones e importaciones hacia/desde el mundo, junto con información arancelaria (AHS y MFN) correspondiente a ese año. La clave primaria natural es la combinación `(Partner Name, Year)`.

**Nota:** El dataset no desglosa por categoría de producto ni por socio comercial específico en este archivo; registra valores totales de comercio bilateral con el mundo.

In [158]:
# Verificación de clave primaria
duplicados_clave = df.duplicated(subset=['Partner Name', 'Year']).sum()
print(f'Combinaciones duplicadas (Partner Name + Year): {duplicados_clave}')
print(f'→ La clave (Partner Name, Year) es única: {duplicados_clave == 0}')

Combinaciones duplicadas (Partner Name + Year): 0
→ La clave (Partner Name, Year) es única: True


---
## 4. Diccionario de datos

La siguiente tabla describe cada variable del dataset original, su tipo, rol analítico y observaciones clave.

In [159]:
diccionario = pd.DataFrame([
    ['Partner Name',                          'str',     'Dimensión',  'País o territorio que reporta el flujo comercial',                                    'Incluye el agregado "World". 265 valores únicos.'],
    ['Year',                                  'int64',   'Temporal',   'Año calendario del flujo comercial registrado',                                       'Rango 1988–2021. Clave primaria junto con Partner Name.'],
    ['Export (US$ Thousand)',                 'float64', 'Métrica',    'Valor total de exportaciones en miles de USD corrientes',                             'Sin ajuste por inflación. 20 valores en cero.'],
    ['Import (US$ Thousand)',                 'float64', 'Métrica',    'Valor total de importaciones en miles de USD corrientes',                             'Sin ajuste por inflación. Sin valores en cero.'],
    ['Export Product Share (%)',              'float64', 'Constante',  'Participación del producto en las exportaciones del país',                            'Siempre 100%. Campo uninformativo → eliminar.'],
    ['Import Product Share (%)',              'int64',   'Constante',  'Participación del producto en las importaciones del país',                            'Siempre 100%. Campo uninformativo → eliminar.'],
    ['Revealed comparative advantage',        'float64', 'Constante',  'Índice de ventaja comparativa revelada de Balassa',                                   'Siempre 1.0 cuando disponible. 41.8% nulos → eliminar.'],
    ['World Growth (%)',                      'float64', 'Métrica',    'Variación porcentual del valor de comercio mundial respecto al año anterior',        '45.5% nulos (años tempranos). Idéntico a Country Growth en todos los casos.'],
    ['Country Growth (%)',                    'float64', 'Métrica',    'Variación porcentual del comercio del país respecto al año anterior',                 '45.5% nulos. Mismo valor que World Growth en todos los registros no-nulos.'],
    ['AHS Simple Average (%)',                'float64', 'Métrica',    'Promedio simple de aranceles aplicados (AHS) en %',                                   'AHS = Applied Harmonized System. 0.2% nulos.'],
    ['AHS Weighted Average (%)',              'float64', 'Métrica',    'Promedio ponderado por importaciones de aranceles AHS en %',                         '0.2% nulos. Máximo 197.76%.'],
    ['AHS Total Tariff Lines',                'float64', 'Métrica',    'Número total de líneas arancelarias en el esquema AHS',                              '0.2% nulos.'],
    ['AHS Dutiable Tariff Lines Share (%)',   'float64', 'Métrica',    '% de líneas AHS con arancel positivo',                                               '0.2% nulos.'],
    ['AHS Duty Free Tariff Lines Share (%)',  'float64', 'Métrica',    '% de líneas AHS exentas de arancel',                                                 '0.2% nulos.'],
    ['AHS Specific Tariff Lines Share (%)',   'float64', 'Métrica',    '% de líneas AHS con arancel específico (por unidad)',                                '0.2% nulos.'],
    ['AHS AVE Tariff Lines Share (%)',        'float64', 'Métrica',    '% de líneas AHS con equivalente ad valorem estimado',                                '0.2% nulos.'],
    ['AHS MaxRate (%)',                       'float64', 'Métrica',    'Tasa arancelaria máxima aplicada bajo AHS',                                           '0.2% nulos. Valor extremo: 5000%.'],
    ['AHS MinRate (%)',                       'float64', 'Métrica',    'Tasa arancelaria mínima aplicada bajo AHS',                                           '0.2% nulos. Mínimo 0%.'],
    ['AHS SpecificDuty Imports (US$ Thousand)','float64','Métrica',   'Importaciones sujetas a arancel específico AHS en miles USD',                        '0.19% nulos.'],
    ['AHS Dutiable Imports (US$ Thousand)',   'float64', 'Métrica',    'Importaciones gravadas bajo AHS en miles USD',                                       '0.19% nulos.'],
    ['AHS Duty Free Imports (US$ Thousand)',  'float64', 'Métrica',    'Importaciones libres de arancel AHS en miles USD',                                   '0.19% nulos.'],
    ['MFN Simple Average (%)',                'float64', 'Métrica',    'Promedio simple de aranceles Nación Más Favorecida (MFN)',                            '0.19% nulos. MFN es el arancel base OMC.'],
    ['MFN Weighted Average (%)',              'float64', 'Métrica',    'Promedio ponderado de aranceles MFN',                                                 '0.19% nulos.'],
    ['MFN Total Tariff Lines',                'float64', 'Métrica',    'Número total de líneas arancelarias MFN',                                             '0.19% nulos.'],
    ['MFN Dutiable Tariff Lines Share (%)',   'float64', 'Métrica',    '% de líneas MFN con arancel positivo',                                               '0.19% nulos.'],
    ['MFN Duty Free Tariff Lines Share (%)',  'float64', 'Métrica',    '% de líneas MFN exentas de arancel',                                                 '0.19% nulos.'],
    ['MFN Specific Tariff Lines Share (%)',   'float64', 'Métrica',    '% de líneas MFN con arancel específico',                                             '0.20% nulos.'],
    ['MFN AVE Tariff Lines Share (%)',        'float64', 'Métrica',    '% de líneas MFN con equivalente ad valorem',                                         '0.20% nulos.'],
    ['MFN MaxRate (%)',                       'float64', 'Métrica',    'Tasa arancelaria máxima MFN',                                                         '0.19% nulos. Valor extremo: 5000%.'],
    ['MFN MinRate (%)',                       'float64', 'Métrica',    'Tasa arancelaria mínima MFN',                                                         '0.19% nulos. Siempre 0%.'],
    ['MFN SpecificDuty Imports (US$ Thousand)','float64','Métrica',   'Importaciones sujetas a arancel específico MFN en miles USD',                        '0.19% nulos.'],
    ['MFN Dutiable Imports (US$ Thousand)',   'float64', 'Métrica',    'Importaciones gravadas bajo MFN en miles USD',                                       '0.19% nulos.'],
    ['MFN Duty Free Imports (US$ Thousand)',  'float64', 'Métrica',    'Importaciones libres de arancel MFN en miles USD',                                   '0.19% nulos.'],
], columns=['Variable', 'Tipo Original', 'Rol Analítico', 'Descripción', 'Observaciones'])

diccionario

,Variable,Tipo Original,Rol Analítico,Descripción,Observaciones
0,Partner Name,str,Dimensión,País o territorio que reporta el flujo comercial,"Incluye el agregado ""World"". 265 valores únicos."
1,Year,int64,Temporal,Año calendario del flujo comercial registrado,Rango 1988–2021. Clave primaria junto con Part...
2,Export (US$ Thousand),float64,Métrica,Valor total de exportaciones en miles de USD c...,Sin ajuste por inflación. 20 valores en cero.
3,Import (US$ Thousand),float64,Métrica,Valor total de importaciones en miles de USD c...,Sin ajuste por inflación. Sin valores en cero.
4,Export Product Share (%),float64,Constante,Participación del producto en las exportacione...,Siempre 100%. Campo uninformativo → eliminar.
5,Import Product Share (%),int64,Constante,Participación del producto en las importacione...,Siempre 100%. Campo uninformativo → eliminar.
6,Revealed comparative advantage,float64,Constante,Índice de ventaja comparativa revelada de Balassa,Siempre 1.0 cuando disponible. 41.8% nulos → e...
7,World Growth (%),float64,Métrica,Variación porcentual del valor de comercio mun...,45.5% nulos (años tempranos). Idéntico a Count...
8,Country Growth (%),float64,Métrica,Variación porcentual del comercio del país res...,45.5% nulos. Mismo valor que World Growth en t...
9,AHS Simple Average (%),float64,Métrica,Promedio simple de aranceles aplicados (AHS) en %,AHS = Applied Harmonized System. 0.2% nulos.


---
## 5. Tabla de perfilado completo

In [160]:
# Construcción de la tabla de perfilado
perfil_rows = []

for col in df.columns:
    nulos        = df[col].isnull().sum()
    pct_nulos    = round(nulos / len(df) * 100, 2)
    cardinalidad = df[col].nunique()
    
    if pd.api.types.is_numeric_dtype(df[col]):
        val_min  = round(float(df[col].min()), 2)
        val_max  = round(float(df[col].max()), 2)
        media    = round(float(df[col].mean()), 2)
        desv_std = round(float(df[col].std()), 2)
        ceros    = int((df[col] == 0).sum())
    else:
        val_min  = 'N/A'
        val_max  = 'N/A'
        media    = 'N/A'
        desv_std = 'N/A'
        ceros    = 'N/A'
    
    perfil_rows.append({
        'Variable':       col,
        'Tipo':           str(df[col].dtype),
        'Nulos':          nulos,
        '% Nulos':        pct_nulos,
        'Cardinalidad':   cardinalidad,
        'Min':            val_min,
        'Max':            val_max,
        'Media':          media,
        'Desv. Estándar': desv_std,
        'Valores en 0':   ceros,
    })

tabla_perfil = pd.DataFrame(perfil_rows)
print(f'Tabla de perfilado: {len(tabla_perfil)} variables analizadas')
tabla_perfil

tabla_perfil.to_csv('../../data/processed/tabla_perfilado.csv', index=False)

Tabla de perfilado: 33 variables analizadas


In [161]:
# Resumen estadístico de variables numéricas
print('ESTADÍSTICAS DESCRIPTIVAS — Variables numéricas')
df.describe().T

ESTADÍSTICAS DESCRIPTIVAS — Variables numéricas


,count,mean,std,min,25%,50%,75%,max
Year,"8,096.00","2,004.91",9.71,"1,988.00","1,997.00","2,005.00","2,013.00","2,021.00"
Export (US$ Thousand),"8,096.00","142,119,180.45","992,841,745.23",0.00,"427,426.38","3,719,682.54","25,855,137.89","24,227,432,526.02"
Import (US$ Thousand),"8,096.00","130,521,647.44","907,380,209.54",0.03,"160,133.53","2,053,967.07","21,029,365.26","21,931,213,223.04"
Export Product Share (%),"8,076.00",100.00,0.00,100.00,100.00,100.00,100.00,100.00
Import Product Share (%),"8,096.00",100.00,0.00,100.00,100.00,100.00,100.00,100.00
Revealed comparative advantage,"4,712.00",1.00,0.00,1.00,1.00,1.00,1.00,1.00
World Growth (%),"4,410.00",3.99,10.00,-62.28,-1.44,3.83,9.41,174.00
Country Growth (%),"4,410.00",3.99,10.00,-62.28,-1.44,3.83,9.41,174.00
AHS Simple Average (%),"8,080.00",6.66,3.84,0.00,4.03,6.12,8.48,46.98
AHS Weighted Average (%),"8,080.00",6.08,6.77,0.00,2.10,4.70,8.05,197.76


In [162]:
# Cobertura por año
cobertura_anual = df.groupby('Year').size().reset_index(name='N_registros')
print('REGISTROS POR AÑO')
print(cobertura_anual.to_string(index=False))

REGISTROS POR AÑO
 Year  N_registros
 1988          207
 1989          209
 1990          212
 1991          211
 1992          232
 1993          234
 1994          234
 1995          234
 1996          230
 1997          230
 1998          230
 1999          231
 2000          244
 2001          245
 2002          245
 2003          244
 2004          243
 2005          242
 2006          244
 2007          244
 2008          244
 2009          244
 2010          246
 2011          248
 2012          246
 2013          247
 2014          246
 2015          247
 2016          246
 2017          247
 2018          248
 2019          247
 2020          248
 2021          247


In [163]:
# Cobertura por país (años disponibles)
cobertura_pais = df.groupby('Partner Name')['Year'].count().reset_index()
cobertura_pais.columns = ['País', 'Años_disponibles']
cobertura_pais = cobertura_pais.sort_values('Años_disponibles')

print('PAÍSES CON COBERTURA INCOMPLETA (< 10 años)')
print(cobertura_pais[cobertura_pais['Años_disponibles'] < 10].to_string(index=False))
print()
print(f'Países con cobertura completa (34 años): {(cobertura_pais["Años_disponibles"] == 34).sum()}')
print(f'Países con cobertura >= 10 años       : {(cobertura_pais["Años_disponibles"] >= 10).sum()}')

PAÍSES CON COBERTURA INCOMPLETA (< 10 años)
                          País  Años_disponibles
              Yemen Democratic                 3
    German Democratic Republic                 3
               Pacific Islands                 4
Yugoslavia,FR(Serbia/Montenegr                 4
                  Soviet Union                 4
    Ethiopia(includes Eritrea)                 5
                Czechoslovakia                 5
                       Reunion                 8
                    Martinique                 8
                 French Guiana                 8
                    Guadeloupe                 8
              Saint Barthélemy                 9

Países con cobertura completa (34 años): 191
Países con cobertura >= 10 años       : 253


---
## 6. Identificación de problemas de calidad

Se identificaron los siguientes tipos de problemas en el dataset original:

In [164]:
## HALLAZGO 1 — 5 COLUMNAS SIN VALOR ANALÍTICO (basura confirmada)

# 1a. Constantes ya conocidas
cols_constantes_known = {
    'Export Product Share (%)': df['Export Product Share (%)'].dropna().unique(),
    'Import Product Share (%)': df['Import Product Share (%)'].unique(),
}
for col, vals in cols_constantes_known.items():
    print(f'  [{col}] → siempre {vals} (constante, 0 variabilidad)')

# 1b. RCA
rca_vals = df['Revealed comparative advantage'].dropna().unique()
rca_nulos = df['Revealed comparative advantage'].isnull().sum()
print(f'  [Revealed comparative advantage] → siempre {rca_vals} | {rca_nulos} nulos ({rca_nulos/len(df)*100:.1f}%)')

# 1c. MFN MinRate — siempre 0
mfn_min_vals = df['MFN MinRate (%)'].dropna().unique()
print(f'  [MFN MinRate (%)] → siempre {mfn_min_vals} (constante)')

# 1d. Country Growth == World Growth
mask = df['World Growth (%)'].notna() & df['Country Growth (%)'].notna()
identicas = (df.loc[mask, 'World Growth (%)'] == df.loc[mask, 'Country Growth (%)']).all()
print(f'  [Country Growth (%)] == [World Growth (%)] en los {mask.sum()} registros no-nulos: {identicas}')
print(f'  → Columna completamente redundante con World Growth (%)')

COLS_BASURA = [
    'Export Product Share (%)',
    'Import Product Share (%)',
    'Revealed comparative advantage',
    'MFN MinRate (%)',
    'Country Growth (%)',
]
print(f'\n→ Total columnas a eliminar: {len(COLS_BASURA)}')
print(f'  Dataset pasa de {df.shape[1]} a {df.shape[1] - len(COLS_BASURA)} columnas útiles reales.')

  [Export Product Share (%)] → siempre [100.] (constante, 0 variabilidad)
  [Import Product Share (%)] → siempre [100] (constante, 0 variabilidad)
  [Revealed comparative advantage] → siempre [1.] | 3384 nulos (41.8%)
  [MFN MinRate (%)] → siempre [0.] (constante)
  [Country Growth (%)] == [World Growth (%)] en los 4410 registros no-nulos: True
  → Columna completamente redundante con World Growth (%)

→ Total columnas a eliminar: 5
  Dataset pasa de 33 a 28 columnas útiles reales.


In [165]:
# HALLAZGO 2 " World" CONTAMINA EL ANÁLISIS (34 registros agregados)'

# El nombre tiene un espacio adelante: ' World', no 'World'
world_mask = df['Partner Name'].str.strip() == 'World'
world_df   = df[world_mask]

print(f'Nombre exacto en el dataset: {repr(df.loc[world_mask, "Partner Name"].iloc[0])}')
print(f'Registros agregados mundiales: {len(world_df)}')
print(f'Años cubiertos: {world_df["Year"].min()} – {world_df["Year"].max()}')
print()

# Por qué distorsionan: comparar magnitudes
mundo_export_max = world_df['Export (US$ Thousand)'].max()
pais_export_max  = df[~world_mask]['Export (US$ Thousand)'].max()
pais_export_med  = df[~world_mask]['Export (US$ Thousand)'].median()

print('MAGNITUD DEL PROBLEMA:')
print(f'  Export máximo de " World"       : ${mundo_export_max/1e6:,.0f} M USD')
print(f'  Export máximo país individual   : ${pais_export_max/1e6:,.0f} M USD')
print(f'  Export mediana países           : ${pais_export_med/1e6:,.0f} M USD')
print(f'  Ratio World/mediana países      : {mundo_export_max/pais_export_med:,.0f}x')
print()
print('→ DECISIÓN: separar en dataset_world.csv (referencia) y excluir')
print('  del dataset analítico principal. También limpiar el espacio del nombre.')

Nombre exacto en el dataset: ' World'
Registros agregados mundiales: 34
Años cubiertos: 1988 – 2021

MAGNITUD DEL PROBLEMA:
  Export máximo de " World"       : $24,227 M USD
  Export máximo país individual   : $8,895 M USD
  Export mediana países           : $4 M USD
  Ratio World/mediana países      : 6,559x

→ DECISIÓN: separar en dataset_world.csv (referencia) y excluir
  del dataset analítico principal. También limpiar el espacio del nombre.


In [166]:

## HALLAZGO 3 — MFN SHARES > 100%: ~4,600 REGISTROS ANÓMALOS


# Contexto técnico: MFN Specific y AVE son "equivalentes ad valorem estimados"
# NO son porcentajes de líneas que deben sumar 100. Vienen así de WITS.
problematicas = {
    'MFN Specific Tariff Lines Share (%)': df['MFN Specific Tariff Lines Share (%)'],
    'MFN AVE Tariff Lines Share (%)':      df['MFN AVE Tariff Lines Share (%)'],
}

for col, serie in problematicas.items():
    no_nulos  = serie.notna().sum()
    over100   = (serie > 100).sum()
    pct_over  = over100 / no_nulos * 100
    print(f'  [{col}]')
    print(f'    Registros no-nulos : {no_nulos:,}')
    print(f'    Valores > 100%     : {over100:,} ({pct_over:.1f}%)')
    print(f'    Máximo             : {serie.max():.2f}%')
    print(f'    Mediana            : {serie.median():.2f}%')
    print()



  [MFN Specific Tariff Lines Share (%)]
    Registros no-nulos : 8,080
    Valores > 100%     : 984 (12.2%)
    Máximo             : 1800.00%
    Mediana            : 28.18%

  [MFN AVE Tariff Lines Share (%)]
    Registros no-nulos : 8,080
    Valores > 100%     : 3,641 (45.1%)
    Máximo             : 3860.00%
    Mediana            : 88.05%



DIAGNÓSTICO:
  MFN Specific y AVE son valores ad valorem estimados (AVE = Ad Valorem
  Equivalent). No son porcentajes de líneas arancelarias [0-100].
  Los valores extremos (ej. 3860%) representan tasas equivalentes reales
  de aranceles específicos muy altos. Es comportamiento esperado de WITS.

  AHS Specific y AHS AVE NO tienen este problema (max 69.4% y 68.5%).
  La inconsistencia es entre esquemas AHS vs MFN, no un error del dataset.

DECISIÓN:
  1. Conservar valores originales (documentar limitación).
  2. Crear flag: flag_mfn_ave_extremo = 1 si MFN AVE Share > 100%.
  3. Para PCA: normalizar con MinMaxScaler o excluir estas columnas.

In [167]:
## HALLAZGO 4 — PAÍSES EXTINTOS: ACTIVOS PARA STORYTELLING

extintos_info = {
    'Soviet Union':                    {'sucesor': 'Rusia + 14 repúblicas', 'fin': 1991},
    'German Democratic Republic':      {'sucesor': 'Alemania reunificada',  'fin': 1990},
    'Yugoslavia,FR(Serbia/Montenegr':  {'sucesor': 'Serbia, Montenegro...',  'fin': 1992},
    'Czechoslovakia':                  {'sucesor': 'Rep. Checa + Eslovaquia','fin': 1992},
    'Yemen Democratic':                {'sucesor': 'Yemen unificado',        'fin': 1990},
}

print(f'{"País":<40} {"Registros":>10} {"Años":>15} {"Sucesor":<30}')
print('-' * 100)
for pais, info in extintos_info.items():
    sub = df[df['Partner Name'] == pais]
    if len(sub) > 0:
        anos = f'{sub["Year"].min()}–{sub["Year"].max()}'
        print(f'{pais:<40} {len(sub):>10} {anos:>15} {info["sucesor"]:<30}')



País                                      Registros            Años Sucesor                       
----------------------------------------------------------------------------------------------------
Soviet Union                                      4       1988–1991 Rusia + 14 repúblicas         
German Democratic Republic                        3       1988–1990 Alemania reunificada          
Yugoslavia,FR(Serbia/Montenegr                    4       1988–1991 Serbia, Montenegro...         
Czechoslovakia                                    5       1988–1992 Rep. Checa + Eslovaquia       
Yemen Democratic                                  3       1988–1990 Yemen unificado               



  NO eliminar del dataset analítico principal.
  Crear columna "entity_status": "Activo" / "Extinto" / "Agregado_Mundial"
  Permite storytelling sobre el fin de la Guerra Fría y su impacto
  en el comercio mundial (quiebre estructural 1989-1992).

---
## 7. Reglas de limpieza y transformaciones

A continuación se aplican todas las reglas de limpieza, una por tipo de problema detectado.

In [168]:
df_clean = df.copy()
print(f'Dataset base copiado: {df_clean.shape[0]} filas × {df_clean.shape[1]} columnas')

Dataset base copiado: 8096 filas × 33 columnas


In [169]:
# -----------------------------------------------------------
# REGLA 1 — Eliminar las 5 columnas sin valor analítico
#
# Problema: columnas con varianza cero o completamente redundantes
#           no aportan información y generan ruido en el análisis.
#
# Columnas eliminadas:
#   · Export/Import Product Share (%): siempre 100
#   · Revealed comparative advantage: siempre 1.0 + 41.8% nulos
#   · MFN MinRate (%):                siempre 0.0
#   · Country Growth (%):             idéntica a World Growth (%)
#                                     en los 4,410 registros no-nulos
# -----------------------------------------------------------

COLS_BASURA = [
    'Export Product Share (%)',
    'Import Product Share (%)',
    'Revealed comparative advantage',
    'MFN MinRate (%)',
    'Country Growth (%)',
]

df_clean = df_clean.drop(columns=COLS_BASURA)

print('REGLA 1 — Columnas sin valor analítico eliminadas.')
print(f'  Eliminadas  : {COLS_BASURA}')
print(f'  Columnas restantes: {df_clean.shape[1]} (antes: {df.shape[1]})')

REGLA 1 — Columnas sin valor analítico eliminadas.
  Eliminadas  : ['Export Product Share (%)', 'Import Product Share (%)', 'Revealed comparative advantage', 'MFN MinRate (%)', 'Country Growth (%)']
  Columnas restantes: 28 (antes: 33)


In [170]:
# -----------------------------------------------------------
# REGLA 2 — Limpiar espacio en Partner Name y separar ' World'
#
# Problema A: El registro del agregado mundial tiene un espacio
#             adelante (' World'), lo que lo hace invisible a
#             filtros directos y rompe joins con tablas externas.
#
# Problema B: Los 34 registros de ' World' son outliers extremos
#             (su Export máximo es ~600,000x la mediana de países)
#             que distorsionan escalas, medias y visualizaciones.
#
# Decisión: strip() en toda la columna + separar World en un
#           dataset de referencia independiente.
# -----------------------------------------------------------

# Limpiar espacios en todos los nombres
df_clean['Partner Name'] = df_clean['Partner Name'].str.strip()

# Separar el agregado mundial
df_world = df_clean[df_clean['Partner Name'] == 'World'].copy()
df_clean = df_clean[df_clean['Partner Name'] != 'World'].copy()

print('REGLA 2 — Espacio limpiado y agregado mundial separado.')
print(f'  Nombre original del agregado : {repr(df["Partner Name"].iloc[0])} → limpiado a "World"')
print(f'  Registros separados en df_world  : {len(df_world)}')
print(f'  Registros en dataset analítico   : {len(df_clean):,}')

REGLA 2 — Espacio limpiado y agregado mundial separado.
  Nombre original del agregado : 'Aruba' → limpiado a "World"
  Registros separados en df_world  : 34
  Registros en dataset analítico   : 8,062


In [171]:
# -----------------------------------------------------------
# REGLA 3 — Clasificar entidades históricas (entity_status)
#           *** DEBE IR ANTES de cualquier exclusión por cobertura ***
#
# Problema: los países extintos tienen < 10 años de datos por
#           naturaleza histórica (no por defecto del dataset),
#           y son valiosos para storytelling del periodo 1989-1992.
#
# Decisión: crear columna entity_status antes de excluir por
#           cobertura, para no perder estos registros.
# -----------------------------------------------------------

PAISES_EXTINTOS = [
    'Soviet Union',                   # disuelta 1991
    'German Democratic Republic',     # reunificación 1990
    'Yugoslavia,FR(Serbia/Montenegr', # disuelta 1992
    'Czechoslovakia',                 # disuelta 1992
    'Yemen Democratic',               # unificación 1990
    'Pacific Islands',                # Trust Territory disuelto 1991
    'Ethiopia(includes Eritrea)',      # separación 1993
]

df_clean['entity_status'] = df_clean['Partner Name'].apply(
    lambda x: 'Extinto' if x in PAISES_EXTINTOS else 'Activo'
)

print('REGLA 3 — entity_status asignado.')
print(f'  entity_status = "Extinto" : {(df_clean["entity_status"]=="Extinto").sum()} registros')
print(f'  entity_status = "Activo"  : {(df_clean["entity_status"]=="Activo").sum():,} registros')
print()
print('  Países extintos conservados para storytelling:')
resumen_extintos = (df_clean[df_clean['entity_status'] == 'Extinto']
                    .groupby('Partner Name')['Year']
                    .agg(['count', 'min', 'max'])
                    .rename(columns={'count': 'N', 'min': 'Desde', 'max': 'Hasta'}))
print(resumen_extintos.to_string())

REGLA 3 — entity_status asignado.
  entity_status = "Extinto" : 28 registros
  entity_status = "Activo"  : 8,034 registros

  Países extintos conservados para storytelling:
                                N  Desde  Hasta
Partner Name                                   
Czechoslovakia                  5   1988   1992
Ethiopia(includes Eritrea)      5   1988   1992
German Democratic Republic      3   1988   1990
Pacific Islands                 4   1988   1991
Soviet Union                    4   1988   1991
Yemen Democratic                3   1988   1990
Yugoslavia,FR(Serbia/Montenegr  4   1988   1991


In [172]:
# -----------------------------------------------------------
# REGLA 4 — Excluir territorios con cobertura insuficiente
#           que NO tienen valor analítico ni histórico
#
# Problema: territorios de ultramar franceses (Guadalupe,
#           Martinica, Reunión, Guayana Francesa, Saint Barthélemy)
#           aparecen con 8 registros o menos. A diferencia de los
#           países extintos, no aportan storytelling relevante
#           y fragmentan el análisis longitudinal.
#
# Criterio: < 10 años de cobertura AND no en PAISES_EXTINTOS.
# -----------------------------------------------------------

cobertura = df_clean.groupby('Partner Name')['Year'].count()

EXCLUIR_COBERTURA = cobertura[
    (cobertura < 10) & (~cobertura.index.isin(PAISES_EXTINTOS))
].index.tolist()

n_antes = len(df_clean)
df_clean = df_clean[~df_clean['Partner Name'].isin(EXCLUIR_COBERTURA)].copy()

print('REGLA 4 — Territorios con cobertura insuficiente excluidos.')
print(f'  Territorios excluidos : {EXCLUIR_COBERTURA}')
print(f'  Filas eliminadas      : {n_antes - len(df_clean)}')
print(f'  Filas restantes       : {len(df_clean):,}')

REGLA 4 — Territorios con cobertura insuficiente excluidos.
  Territorios excluidos : ['French Guiana', 'Guadeloupe', 'Martinique', 'Reunion', 'Saint Barthélemy']
  Filas eliminadas      : 41
  Filas restantes       : 8,021


In [173]:
# -----------------------------------------------------------
# REGLA 5 — Flag de exportaciones en cero
#
# Problema: 20 registros con Export = 0 son probables datos
#           faltantes codificados como cero, o territorios sin
#           actividad exportadora formal registrada. No es posible
#           distinguir ambos casos sin fuente externa.
#
# Decisión: conservar el valor (no imputar) y crear flag binario
#           para que el analista los filtre en Tableau según contexto.
# -----------------------------------------------------------

df_clean['flag_export_cero'] = (df_clean['Export (US$ Thousand)'] == 0).astype(int)

print('REGLA 5 — Flag de exportaciones en cero creado.')
print(f'  flag_export_cero = 1 : {df_clean["flag_export_cero"].sum()} registros')
print()
print('  Detalle de registros afectados:')
print(df_clean[df_clean['flag_export_cero'] == 1][
    ['Partner Name', 'Year', 'Export (US$ Thousand)', 'Import (US$ Thousand)']
].to_string(index=False))

REGLA 5 — Flag de exportaciones en cero creado.
  flag_export_cero = 1 : 20 registros

  Detalle de registros afectados:
        Partner Name  Year  Export (US$ Thousand)  Import (US$ Thousand)
      Western Sahara  1988                   0.00                 172.74
      Br. Antr. Terr  1991                   0.00                  37.50
        Neutral Zone  2000                   0.00                  12.94
        Us Msc.Pac.I  2000                   0.00                   9.93
      Br. Antr. Terr  2001                   0.00                  22.31
        Neutral Zone  2001                   0.00                   4.92
              Monaco  2002                   0.00                 200.29
        Us Msc.Pac.I  2002                   0.00                   3.99
      Br. Antr. Terr  2003                   0.00                   1.33
        Neutral Zone  2003                   0.00                  10.44
        Neutral Zone  2008                   0.00                   0.38
   

In [174]:
# -----------------------------------------------------------
# REGLA 6 — Flag de valores MFN AVE extremos (> 100%)
#
# Problema: MFN AVE Tariff Lines Share (%) supera 100% en 3,627
#           registros (45% del dataset), con máximo de 3,860%.
#           Idem MFN Specific Share: 984 registros > 100%.
#
# Diagnóstico: estas columnas son equivalentes ad valorem estimados
#              (Ad Valorem Equivalent), NO porcentajes de líneas
#              [0-100]. Valores altos son reales en aranceles
#              específicos (por cantidad). Fuente: metodología WITS.
#              AHS equivalentes no tienen este problema (max 68.5%).
#
# Decisión: conservar valores originales. Crear flag para excluir
#           estas columnas en PCA o modelos que asuman [0-100].
# -----------------------------------------------------------

df_clean['flag_mfn_ave_extremo'] = (
    df_clean['MFN AVE Tariff Lines Share (%)'] > 100
).astype(int)

total_no_nulos = df_clean['MFN AVE Tariff Lines Share (%)'].notna().sum()
afectados      = df_clean['flag_mfn_ave_extremo'].sum()

print('REGLA 6 — Flag de MFN AVE extremo creado.')
print(f'  flag_mfn_ave_extremo = 1 : {afectados:,} registros ({afectados/total_no_nulos*100:.1f}% de no-nulos)')
print(f'  MFN AVE Share — máximo   : {df_clean["MFN AVE Tariff Lines Share (%)"].max():.0f}%')
print(f'  MFN AVE Share — mediana  : {df_clean["MFN AVE Tariff Lines Share (%)"].median():.1f}%')
print()
print('  Nota: AHS AVE Share (mismo concepto) no tiene este problema.')
print(f'  AHS AVE Share — máximo   : {df_clean["AHS AVE Tariff Lines Share (%)"].max():.1f}%')

REGLA 6 — Flag de MFN AVE extremo creado.
  flag_mfn_ave_extremo = 1 : 3,627 registros (45.3% de no-nulos)
  MFN AVE Share — máximo   : 3860%
  MFN AVE Share — mediana  : 89.0%

  Nota: AHS AVE Share (mismo concepto) no tiene este problema.
  AHS AVE Share — máximo   : 68.5%


In [175]:
# -----------------------------------------------------------
# REGLA 7 — Crear variables derivadas (métricas analíticas)
#
# Problema: los datos en miles de USD dificultan la lectura
#           en dashboards. Además, faltan KPIs centrales del
#           análisis de comercio exterior.
#
# Variables creadas:
#   · Export/Import en millones (legibilidad)
#   · Trade Balance = Export - Import (KPI de superávit/déficit)
#   · Total Trade = Export + Import (apertura comercial)
#   · Trade Status: categoría texto para filtros en Tableau
# -----------------------------------------------------------

df_clean['Export (US$ Million)']         = (df_clean['Export (US$ Thousand)'] / 1000).round(3)
df_clean['Import (US$ Million)']         = (df_clean['Import (US$ Thousand)'] / 1000).round(3)
df_clean['Trade Balance (US$ Thousand)'] = df_clean['Export (US$ Thousand)'] - df_clean['Import (US$ Thousand)']
df_clean['Trade Balance (US$ Million)']  = (df_clean['Trade Balance (US$ Thousand)'] / 1000).round(3)
df_clean['Total Trade (US$ Thousand)']   = df_clean['Export (US$ Thousand)'] + df_clean['Import (US$ Thousand)']
df_clean['Total Trade (US$ Million)']    = (df_clean['Total Trade (US$ Thousand)'] / 1000).round(3)
df_clean['Trade Status'] = df_clean['Trade Balance (US$ Thousand)'].apply(
    lambda x: 'Superávit' if x > 0 else ('Déficit' if x < 0 else 'Equilibrio')
)

print('REGLA 7 — Variables derivadas creadas.')
print('  · Export (US$ Million)           → Export (US$ Thousand) / 1,000')
print('  · Import (US$ Million)           → Import (US$ Thousand) / 1,000')
print('  · Trade Balance (US$ Thousand)   → Export - Import')
print('  · Trade Balance (US$ Million)    → Trade Balance / 1,000')
print('  · Total Trade (US$ Thousand)     → Export + Import')
print('  · Total Trade (US$ Million)      → Total Trade / 1,000')
print('  · Trade Status                   → "Superávit" / "Déficit" / "Equilibrio"')
print()
print(f'  Distribución Trade Status:')
print(df_clean['Trade Status'].value_counts().to_string())

REGLA 7 — Variables derivadas creadas.
  · Export (US$ Million)           → Export (US$ Thousand) / 1,000
  · Import (US$ Million)           → Import (US$ Thousand) / 1,000
  · Trade Balance (US$ Thousand)   → Export - Import
  · Trade Balance (US$ Million)    → Trade Balance / 1,000
  · Total Trade (US$ Thousand)     → Export + Import
  · Total Trade (US$ Million)      → Total Trade / 1,000
  · Trade Status                   → "Superávit" / "Déficit" / "Equilibrio"

  Distribución Trade Status:
Trade Status
Superávit    5737
Déficit      2284


In [176]:
# -----------------------------------------------------------
# REGLA 8 — Corrección de tipos y ordenamiento final
#
# Problema: Year debe ser integer explícito para que Tableau
#           no lo interprete como medida continua (float).
#           El orden por (País, Año) facilita series temporales.
# -----------------------------------------------------------

df_clean['Year'] = df_clean['Year'].astype(int)
df_clean = df_clean.sort_values(['Partner Name', 'Year']).reset_index(drop=True)

print('REGLA 8 — Tipos corregidos y dataset ordenado.')
print(f'  Year dtype: {df_clean["Year"].dtype}  (int64 → Tableau lo trata como dimensión discreta)')
print(f'  Orden: (Partner Name ASC, Year ASC)')
print(f'  Índice reseteado.')
print(f'  Shape final: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas')

REGLA 8 — Tipos corregidos y dataset ordenado.
  Year dtype: int64  (int64 → Tableau lo trata como dimensión discreta)
  Orden: (Partner Name ASC, Year ASC)
  Índice reseteado.
  Shape final: 8,021 filas × 38 columnas


---
## 8. Bitácora de transformaciones

La siguiente tabla documenta todas las decisiones de limpieza, distinguiendo el **dato original** del **dato transformado** y justificando cada decisión.

In [177]:
bitacora = pd.DataFrame([
    [
        'B-001',
        'Columnas constantes',
        'Export Product Share (%), Import Product Share (%), Revealed comparative advantage',
        'Columnas con único valor (100 o 1.0). No aportan información.',
        'Eliminación',
        'Columnas presentes con valor único',
        'Columnas eliminadas del dataset limpio',
        'Dato original preservado en CSV fuente',
        'Alta'
    ],
    [
        'B-002',
        'Entidades con cobertura insuficiente',
        f'{len(PAISES_EXCLUIDOS)} países: {PAISES_EXCLUIDOS[:4]}...',
        'Países extintos o territorios con < 10 años de datos. Impiden análisis longitudinal.',
        'Exclusión de filas',
        f'{filas_antes} filas (con entidades incompletas)',
        f'{filas_despues} filas (sin entidades incompletas)',
        'Registros excluidos no eliminados del CSV original',
        'Alta'
    ],
    [
        'B-003',
        'Exportaciones en cero',
        'Export (US$ Thousand) == 0 (20 registros)',
        'Posibles valores faltantes codificados como cero o territorios sin exportaciones.',
        'Flag binario (flag_export_cero)',
        'Export = 0 sin distinción de causa',
        'flag_export_cero = 1 para registros afectados',
        'Valor cero conservado; filtrable en Tableau con el flag',
        'Media'
    ],
    [
        'B-004',
        'Escala de valores monetarios',
        'Export/Import en miles de USD (difícil lectura)',
        'Los valores en miles dificultan la lectura en dashboards.',
        'Creación de variables derivadas en millones',
        'Export (US$ Thousand)',
        'Export (US$ Million), Import (US$ Million)',
        'Columnas originales en miles conservadas',
        'Baja'
    ],
    [
        'B-005',
        'Ausencia de métrica de balanza comercial',
        'No existía columna de diferencia Export - Import',
        'La balanza es un KPI central del análisis de comercio exterior.',
        'Creación de variable derivada',
        'No existía',
        'Trade Balance (US$ Thousand y Million) + Trade Status',
        'Calculada como Export - Import',
        'Media'
    ],
    [
        'B-006',
        'Ausencia de comercio total',
        'No existía columna de suma bilateral',
        'El comercio total (X + M) es una métrica de apertura comercial.',
        'Creación de variable derivada',
        'No existía',
        'Total Trade (US$ Thousand y Million)',
        'Calculada como Export + Import',
        'Baja'
    ],
    [
        'B-007',
        'World Growth == Country Growth (redundancia)',
        'Ambas columnas son idénticas en todos los registros no-nulos',
        'Aparente error de codificación en la fuente. Ambas reflejan crecimiento mundial.',
        'Documentación (sin eliminación)',
        'Dos columnas independientes con valores idénticos',
        'Ambas conservadas con nota en diccionario',
        'Se recomienda validar con fuente original (UNCTAD/WTO)',
        'Media'
    ],
    [
        'B-008',
        'Valores nominales sin ajuste por inflación',
        'Export/Import en USD corrientes (no constantes)',
        'Comparar 1988 con 2021 en valores nominales sobreestima el crecimiento real.',
        'Documentación y uso de métricas relativas',
        'USD corrientes',
        'Advertencia documentada; métricas relativas prioritarias en dashboard',
        'Limitación intrínseca del dataset. No corregible sin deflactor externo.',
        'Alta'
    ],
    [
        'B-009',
        'Orden del dataset',
        'Orden original sin estructura definida',
        'El orden por (País, Año) facilita análisis temporal y conexión a Tableau.',
        'Reordenamiento',
        'Orden original',
        'Ordenado por (Partner Name ASC, Year ASC)',
        'Operación reversible; no modifica valores',
        'Baja'
    ],
], columns=[
    'ID',
    'Tipo de Problema',
    'Campo(s) Afectado(s)',
    'Descripción del Problema',
    'Acción Aplicada',
    'Dato Original',
    'Dato Transformado',
    'Trazabilidad',
    'Impacto'
])

print(f'Bitácora de transformaciones: {len(bitacora)} decisiones registradas')
bitacora

bitacora.to_csv('../../data/processed/bitacora_transformaciones.csv', index=False)

Bitácora de transformaciones: 9 decisiones registradas


---
## 9. Dataset limpio — Validación y exportación

In [178]:
print('VALIDACIÓN DEL DATASET LIMPIO')
print('=' * 60)
print(f'  Filas           : {df_clean.shape[0]:,}')
print(f'  Columnas        : {df_clean.shape[1]}')
print(f'  Duplicados      : {df_clean.duplicated(subset=["Partner Name","Year"]).sum()}')
print(f'  Países únicos   : {df_clean["Partner Name"].nunique()}')
print(f'  Rango temporal  : {df_clean["Year"].min()} – {df_clean["Year"].max()}')
print()

print('TIPOS DE DATOS — Dataset limpio')
print(df_clean.dtypes.to_string())

VALIDACIÓN DEL DATASET LIMPIO
  Filas           : 8,021
  Columnas        : 38
  Duplicados      : 0
  Países únicos   : 259
  Rango temporal  : 1988 – 2021

TIPOS DE DATOS — Dataset limpio
Partner Name                                   str
Year                                         int64
Export (US$ Thousand)                      float64
Import (US$ Thousand)                      float64
World Growth (%)                           float64
AHS Simple Average (%)                     float64
AHS Weighted Average (%)                   float64
AHS Total Tariff Lines                     float64
AHS Dutiable Tariff Lines Share (%)        float64
AHS Duty Free Tariff Lines Share (%)       float64
AHS Specific Tariff Lines Share (%)        float64
AHS AVE Tariff Lines Share (%)             float64
AHS MaxRate (%)                            float64
AHS MinRate (%)                            float64
AHS SpecificDuty Imports (US$ Thousand)    float64
AHS Dutiable Imports (US$ Thousand)        fl

In [179]:
print('RESUMEN DE NULOS — Variables críticas para Tableau')
cols_criticas = [
    'Partner Name', 'Year',
    'Export (US$ Million)', 'Import (US$ Million)',
    'Trade Balance (US$ Million)', 'Total Trade (US$ Million)',
    'Trade Status', 'flag_export_cero'
]
nulos_criticos = df_clean[cols_criticas].isnull().sum()
print(nulos_criticos.to_string())
print()
print('→ Las variables críticas para el dashboard no tienen nulos.')

RESUMEN DE NULOS — Variables críticas para Tableau
Partner Name                   0
Year                           0
Export (US$ Million)           0
Import (US$ Million)           0
Trade Balance (US$ Million)    0
Total Trade (US$ Million)      0
Trade Status                   0
flag_export_cero               0

→ Las variables críticas para el dashboard no tienen nulos.


In [180]:
print('MUESTRA DEL DATASET LIMPIO (primeras 5 filas)')
df_clean[[
    'Partner Name', 'Year',
    'Export (US$ Million)', 'Import (US$ Million)',
    'Trade Balance (US$ Million)', 'Total Trade (US$ Million)',
    'Trade Status', 'flag_export_cero'
]].head()

MUESTRA DEL DATASET LIMPIO (primeras 5 filas)


,Partner Name,Year,Export (US$ Million),Import (US$ Million),Trade Balance (US$ Million),Total Trade (US$ Million),Trade Status,flag_export_cero
0,Afghanistan,1988,213.03,54.46,158.57,267.49,Superávit,0
1,Afghanistan,1989,299.95,48.63,251.32,348.58,Superávit,0
2,Afghanistan,1990,322.14,55.66,266.48,377.81,Superávit,0
3,Afghanistan,1991,282.83,62.52,220.31,345.35,Superávit,0
4,Afghanistan,1992,291.79,47.73,244.05,339.52,Superávit,0


In [181]:
# Verificación de compatibilidad con Tableau
print('VERIFICACIÓN DE COMPATIBILIDAD CON TABLEAU')
print('=' * 60)

problemas_tableau = []
for col in df_clean.columns:
    if df_clean[col].dtype == object:
        tipo_tableau = 'String'
    elif df_clean[col].dtype == 'int64':
        tipo_tableau = 'Integer'
    elif df_clean[col].dtype == 'float64':
        tipo_tableau = 'Float'
    else:
        tipo_tableau = 'REVISAR'
        problemas_tableau.append(col)

if problemas_tableau:
    print(f'⚠ Columnas con tipo ambiguo: {problemas_tableau}')
else:
    print('✓ Todos los tipos de datos son compatibles con Tableau (String / Integer / Float).')
    print('✓ No hay ambigüedad de tipos.')
    print('✓ El dataset puede conectarse directamente como fuente de datos.')

VERIFICACIÓN DE COMPATIBILIDAD CON TABLEAU
⚠ Columnas con tipo ambiguo: ['Partner Name', 'entity_status', 'Trade Status']


In [182]:
# Exportar dataset limpio
import os
df_clean.to_csv(RUTA_LIMPIO, index=False, encoding='utf-8-sig')

print(f'Dataset limpio exportado exitosamente:')
print(f'  Ruta   : {RUTA_LIMPIO}')
print(f'  Filas  : {df_clean.shape[0]:,}')
print(f'  Cols   : {df_clean.shape[1]}')
print(f'  Enc    : UTF-8 con BOM (compatible con Excel y Tableau)')

Dataset limpio exportado exitosamente:
  Ruta   : ../../data/processed/dataset_limpio_entrega2.csv
  Filas  : 8,021
  Cols   : 38
  Enc    : UTF-8 con BOM (compatible con Excel y Tableau)


---
## 10. Modelado de datos

### Estructura lógica del dataset limpio

El dataset limpio sigue un modelo de **tabla plana desnormalizada** (flat table), adecuado para Tableau sin necesidad de joins adicionales.

```
┌─────────────────────────────────────────────────────────────────┐
│              TABLA PRINCIPAL: comercio_mundial_limpio            │
│            (Granularidad: 1 fila = 1 país × 1 año)              │
├─────────────────────────────┬───────────────────────────────────┤
│  CLAVE PRIMARIA             │  DIMENSIONES                      │
│  Partner Name (PK)          │  Trade Status                     │
│  Year (PK)                  │  flag_export_cero                 │
│                             │  is_deficit / is_surplus          │
├─────────────────────────────┼───────────────────────────────────┤
│  MÉTRICAS COMERCIALES       │  MÉTRICAS ARANCELARIAS            │
│  Export (US$ Million)       │  AHS Simple Average (%)           │
│  Import (US$ Million)       │  AHS Weighted Average (%)         │
│  Trade Balance (US$ Million)│  MFN Simple Average (%)           │
│  Total Trade (US$ Million)  │  MFN Weighted Average (%)         │
│  Country Growth (%)         │  AHS/MFN MaxRate (%)              │
│  World Growth (%)           │  ... (demás vars arancelarias)    │
└─────────────────────────────┴───────────────────────────────────┘
```

### Relaciones posibles con fuentes externas

| Join Key | Fuente Externa | Propósito |
|---|---|---|
| `Partner Name` → ISO 3166 | Tabla de países ISO | Normalizar nombres, agregar región/continente |
| `Year` → Indicadores macroeconómicos | WDI (World Bank) | Añadir PIB, población, inflación |
| `Partner Name` + `Year` → UNCTAD | UNCTADstat | Desagregar por categoría de producto |

### Variables para Tableau

| Tipo en Tableau | Variables |
|---|---|
| **Dimensión geográfica** | `Partner Name` |
| **Dimensión temporal** | `Year` |
| **Dimensión categórica** | `Trade Status`, `flag_export_cero` |
| **Medida continua** | `Export`, `Import`, `Trade Balance`, `Total Trade` (todas en USD Million) |
| **Medida arancelaria** | `AHS/MFN Simple Average (%)`, `AHS/MFN Weighted Average (%)`, `MaxRate` |
| **Medida de crecimiento** | `Country Growth (%)`, `World Growth (%)` |

In [183]:
# Resumen final del modelado
print('RESUMEN DEL MODELO DE DATOS')
print('=' * 60)
print(f'  Tipo de modelo        : Tabla plana (flat table)')
print(f'  Granularidad          : País × Año')
print(f'  Clave primaria        : (Partner Name, Year)')
print(f'  Total de variables    : {df_clean.shape[1]}')
print()

grupos = {
    'Identificadores / Dimensiones': ['Partner Name', 'Year', 'Trade Status', 'flag_export_cero'],
    'Métricas comerciales (miles USD)': ['Export (US$ Thousand)', 'Import (US$ Thousand)',
                                          'Trade Balance (US$ Thousand)', 'Total Trade (US$ Thousand)'],
    'Métricas comerciales (millones USD)': ['Export (US$ Million)', 'Import (US$ Million)',
                                             'Trade Balance (US$ Million)', 'Total Trade (US$ Million)'],
    'Métricas de crecimiento': ['World Growth (%)', 'Country Growth (%)'],
    'Métricas arancelarias AHS': [c for c in df_clean.columns if c.startswith('AHS')],
    'Métricas arancelarias MFN': [c for c in df_clean.columns if c.startswith('MFN')],
}

for grupo, cols in grupos.items():
    print(f'  {grupo}: {len(cols)} variables')

print()
print('Dataset listo para conectar a Tableau.')

RESUMEN DEL MODELO DE DATOS
  Tipo de modelo        : Tabla plana (flat table)
  Granularidad          : País × Año
  Clave primaria        : (Partner Name, Year)
  Total de variables    : 38

  Identificadores / Dimensiones: 4 variables
  Métricas comerciales (miles USD): 4 variables
  Métricas comerciales (millones USD): 4 variables
  Métricas de crecimiento: 2 variables
  Métricas arancelarias AHS: 12 variables
  Métricas arancelarias MFN: 11 variables

Dataset listo para conectar a Tableau.


In [184]:
# Vista final del dataset limpio
print('COLUMNAS FINALES DEL DATASET LIMPIO')
for i, col in enumerate(df_clean.columns, 1):
    print(f'  {i:02d}. {col}  [{df_clean[col].dtype}]')

COLUMNAS FINALES DEL DATASET LIMPIO
  01. Partner Name  [str]
  02. Year  [int64]
  03. Export (US$ Thousand)  [float64]
  04. Import (US$ Thousand)  [float64]
  05. World Growth (%)  [float64]
  06. AHS Simple Average (%)  [float64]
  07. AHS Weighted Average (%)  [float64]
  08. AHS Total Tariff Lines  [float64]
  09. AHS Dutiable Tariff Lines Share (%)  [float64]
  10. AHS Duty Free Tariff Lines Share (%)  [float64]
  11. AHS Specific Tariff Lines Share (%)  [float64]
  12. AHS AVE Tariff Lines Share (%)  [float64]
  13. AHS MaxRate (%)  [float64]
  14. AHS MinRate (%)  [float64]
  15. AHS SpecificDuty Imports (US$ Thousand)  [float64]
  16. AHS Dutiable Imports (US$ Thousand)  [float64]
  17. AHS Duty Free Imports (US$ Thousand)  [float64]
  18. MFN Simple Average (%)  [float64]
  19. MFN Weighted Average (%)  [float64]
  20. MFN Total Tariff Lines  [float64]
  21. MFN Dutiable Tariff Lines Share (%)  [float64]
  22. MFN Duty Free Tariff Lines Share (%)  [float64]
  23. MFN Specifi